# PerfuSense AI · Lesson 6: the final model and the posture service

Everything so far was **measuring**. Now we finish the job:

1. Put everything we learned into one recipe.
2. Compare all the models fairly, with the same head start.
3. Choose the final model from the evidence.
4. Train it once on all the people and **save it to a file**.
5. Check that the saved file gives the same answers.
6. Serve it with FastAPI, so the PerfuSense server can ask for the posture.

After this, nothing is trained again unless the data changes. The service only loads the file.

> Research prototype, not a medical device. This model has **never been tested on our Velostat mat**.

In [1]:
import sys
import inspect
import json
import os
import time

sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import keras

from perfusense_ai import data as D
from perfusense_ai import model as M


def show(*functions):
    """Print the real source code of our shared functions."""
    for f in functions:
        print(inspect.getsource(f))

## Step 1 · The final dataset

Everything we learned goes into one recipe:

| Lesson | What goes in |
|---|---|
| 1 | clean frames, 30-frame windows, per-window scaling, split by person |
| 2 | both real datasets, in one view |
| 3 | the 30×30 cm patch, with placement error |
| 4 | PoPu as full sessions, so training includes real turns |
| 5 | start from synthetic pre-training |

Classes are supine, left, right. Prone seconds stay inside the windows as history, but no window ends on prone.

In [2]:
recordings = D.load_pmd() + [r for r in D.load_popu() if r["label"] != "prone"]
positions = D.mat_positions(recordings)
rng = np.random.default_rng(0)
view = D.patch_view(positions, jitter=2, rng=rng)

Xp, yp, pp = D.windows_from_recordings([r for r in recordings if r["dataset"] == "pmd"], view)
del recordings

sessions = [D.session_windows(p, view, classes=D.CLASSES) for p in range(1, 61)]
X = np.concatenate([Xp] + [s[0] for s in sessions])
y = np.concatenate([yp] + [s[1] for s in sessions])
moving = np.concatenate([np.zeros(len(yp), dtype=bool)] + [s[2] for s in sessions])
people = np.concatenate([pp] + [[f"popu-{i + 1}"] * len(s[1]) for i, s in enumerate(sessions)])
source = np.array([p.split("-")[0] for p in people])
del Xp, sessions
print("real windows:", X.shape, "| people:", len(set(people)), "| moving windows:", int(moving.sum()))
print("per class:", dict(zip(D.CLASSES, np.bincount(y))))

real windows: (5834, 30, 10, 10) | people: 73 | moving windows: 1359
per class: {'supine': np.int64(2182), 'left': np.int64(1788), 'right': np.int64(1864)}


## Step 2 · Pre-train every model the same way

Lesson 5 pre-trained only the Transformer. A fair final comparison gives all three models the same head start. Here the pre-training uses **synthetic data only**, with 1,000 synthetic windows for early stopping, so no real person influences it.

In [3]:
show(D.synthetic_patch_windows)

X_synth, y_synth = D.synthetic_patch_windows(np.random.default_rng(1))
order = np.random.default_rng(2).permutation(len(y_synth))
hold, rest = order[:1000], order[1000:]

BUILDERS = {"baseline": M.build_baseline, "last-frames": M.build_last_frames, "transformer": M.build_transformer}
os.makedirs("../models", exist_ok=True)

def weights_path(name):
    return f"../models/pretrained_{name}.weights.h5"

for name, builder in BUILDERS.items():
    keras.backend.clear_session()
    keras.utils.set_random_seed(42)
    t0 = time.time()
    m = builder()
    M.train(m, X_synth[rest], y_synth[rest], X_synth[hold], y_synth[hold], epochs=20, batch_size=64, patience=3)
    m.save_weights(weights_path(name))
    print(f"{name}: pre-trained in {time.time() - t0:.0f} s")
del X_synth

def synthetic_patch_windows(rng, jitter=2):
    """All steps of Lesson 5 in one call: X (n, 30, 10, 10) and class index (n,)."""
    frames, labels = load_bodypressure(mirror=True)
    padded = pad_to_width(frames)
    top, left = sacrum_point(padded[labels == 0].mean(axis=0))
    patches = np.stack([crop_patch(f[None], top + rng.integers(-jitter, jitter + 1),
                                   left + rng.integers(-jitter, jitter + 1))[0] for f in padded])
    return still_windows(patches, rng), labels



baseline: pre-trained in 10 s


last-frames: pre-trained in 9 s


transformer: pre-trained in 113 s


## Step 3 · The final comparison

Six runs, each 5-fold by person: three models, from zero and pre-trained.

In [4]:
def evaluate(name, pretrained):
    def build():
        m = BUILDERS[name]()
        if pretrained:
            m.load_weights(weights_path(name))
        return m

    t0 = time.time()
    pred = M.cross_validate(build, X, y, people, lr=3e-4 if pretrained else 1e-3)
    side = np.isin(y, [1, 2])
    swapped = side & np.isin(pred, [1, 2]) & (y != pred)
    return {"model": name, "start": "pre-trained" if pretrained else "from zero",
            "all": M.score(y, pred)["balanced_acc"], "macro F1": M.score(y, pred)["macro_f1"],
            "still": M.score(y[~moving], pred[~moving])["balanced_acc"],
            "moving": M.score(y[moving], pred[moving])["balanced_acc"],
            "PmatData": M.score(y[source == "pmd"], pred[source == "pmd"])["balanced_acc"],
            "PoPu": M.score(y[source == "popu"], pred[source == "popu"])["balanced_acc"],
            "left/right swapped": round(float(swapped.sum() / side.sum()), 3),
            "seconds": round(time.time() - t0)}

final_table = pd.DataFrame([evaluate(name, pretrained) for name in BUILDERS for pretrained in (False, True)])
final_table

,model,start,all,macro F1,still,moving,PmatData,PoPu,left/right swapped,seconds
0,baseline,from zero,0.739,0.739,0.769,0.664,0.538,0.791,0.170,35
1,baseline,pre-trained,0.764,0.764,0.810,0.645,0.587,0.809,0.148,51
2,last-frames,from zero,0.808,0.807,0.773,0.913,0.562,0.868,0.113,37
3,last-frames,pre-trained,0.847,0.847,0.817,0.942,0.619,0.901,0.094,56
4,transformer,from zero,0.801,0.802,0.772,0.907,0.606,0.852,0.119,176
5,transformer,pre-trained,0.808,0.807,0.801,0.840,0.605,0.860,0.121,183


### Reading the table

Balanced accuracy, 73 people, each tested unseen, 30×30 cm patch with placement error:

| Model | Start | All | Still | Moving | PmatData | PoPu | Left/right swapped |
|---|---|---|---|---|---|---|---|
| baseline | from zero | 73.9 % | 76.9 % | 66.4 % | 53.8 % | 79.1 % | 17.0 % |
| baseline | pre-trained | 76.4 % | 81.0 % | 64.5 % | 58.7 % | 80.9 % | 14.8 % |
| last-frames | from zero | 80.8 % | 77.3 % | 91.3 % | 56.2 % | 86.8 % | 11.3 % |
| **last-frames** | **pre-trained** | **84.7 %** | **81.7 %** | **94.2 %** | **61.9 %** | **90.1 %** | **9.4 %** |
| Transformer | from zero | 80.1 % | 77.2 % | 90.7 % | 60.6 % | 85.2 % | 11.9 % |
| Transformer | pre-trained | 80.8 % | 80.1 % | 84.0 % | 60.5 % | 86.0 % | 12.1 % |

**The Transformer did not win.** The simple last-frames model with pre-training is the best on almost every column, and it is 11 times smaller.

Look at what pre-training did to the Transformer. It helped on still windows (77.2 → 80.1 %) but **hurt on moving windows (90.7 → 84.0 %)**. A likely reason: the synthetic windows are one frame repeated 30 times, so the Transformer learns "all seconds look alike", which is wrong during a turn. The last-frames model only ever looks at the newest 5 frames, so it has nothing to unlearn. Lesson 5's big gain came from still windows, and a fair test on moving data shows its limit.

### Decision: ship the model that won

The team plan says: *"Baseline first; the Transformer only if it beats the baseline on people it never saw."* It did beat the plan's averaging baseline, but it lost to a fair simple competitor. So the final model is **last-frames, pre-trained**: 84.7 % balanced accuracy.

This is not a failed project. You built a Transformer, tested it fairly, and can explain with evidence why a smaller model is better *on this data*. That is exactly what a science fair judge wants to see. The Transformer may still win on our own mat, with longer turning patterns, or if pre-training includes fake turns. To serve it instead, change one line below: `FINAL = "transformer"`.

A bonus: 6,659 parameters is about 27 KB, small enough that it might even run on the ESP32. That has not been tested.

## Step 4 · Train on everyone

An important difference between the last two steps of any project:

| | Lessons 1–5 and Step 3 | Final model |
|---|---|---|
| Purpose | **measure** how good the method is | **use** it |
| Data | some people always held out | everyone, because more people is better |
| Its accuracy | measured by cross-validation | **not measured**: it has no unseen people left |

So the number to report for this file is the cross-validated one from Step 3, not a test on its own training people. We keep only 6 people out, for early stopping.

In [5]:
keep_out = np.random.default_rng(2).choice(sorted(set(people)), 6, replace=False)
va = np.isin(people, keep_out)

FINAL = "last-frames"     # the winner of Step 3. Use "transformer" to serve the Transformer instead

keras.backend.clear_session()
keras.utils.set_random_seed(42)
final = BUILDERS[FINAL]()
final.load_weights(weights_path(FINAL))

t0 = time.time()
M.train(final, X[~va], y[~va], X[va], y[va], batch_size=64, lr=3e-4)
print(f"trained in {time.time() - t0:.0f} s")

trained in 10 s


## Step 5 · Save it, load it, compare

Two files go into `ai/models/`:

- `posture.keras`: the weights and the layers;
- `posture.json`: what the service must know to use the model correctly. Without it, someone could feed the wrong shape or read the classes in the wrong order.

If you choose the Transformer, its custom layers are registered with `@keras.saving.register_keras_serializable`, which is why Keras can rebuild them from the file.

In [6]:
final.save("../models/posture.keras")
meta = {"name": f"posture-{FINAL}-v1", "classes": D.CLASSES, "t": D.T, "grid": D.GRID,
        "input": "30 frames at 1 Hz, 100 mat bytes each, row-major, as the server stores them",
        "trained_on": sorted({p.split("-")[0] for p in people}), "people": len(set(people)),
        "tested_on_our_mat": False}
with open("../models/posture.json", "w") as f:
    json.dump(meta, f, indent=2)

loaded = keras.models.load_model("../models/posture.keras")
same = np.allclose(final.predict(X[:64], verbose=0), loaded.predict(X[:64], verbose=0), atol=1e-5)
print("saved file gives the same answers:", same)
print("file size:", round(os.path.getsize("../models/posture.keras") / 1e6, 2), "MB")

saved file gives the same answers: True
file size: 0.11 MB


## Step 6 · The service

`ai/serve.py` is a small FastAPI app. Read it: it is the whole product of this work.

In [7]:
print(open("../serve.py").read())

"""Posture service: the PerfuSense server sends the last 30 frames, this answers with the posture.

Run from the ai folder:   .venv/Scripts/python.exe -m uvicorn serve:app --port 8000

Research prototype, not a medical device. The model has never been tested on our own mat.
"""
import json
import os
from pathlib import Path

import keras
import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

from perfusense_ai import data
from perfusense_ai import model as _layers  # noqa: F401  (registers our custom layers with Keras)

MODEL_DIR = Path(os.environ.get("PERFUSENSE_MODEL_DIR", Path(__file__).parent / "models"))

# PLACEHOLDER: below this average byte per cell, the mat counts as empty. It must be measured on our
# own mat. It is not a clinical value. We use the average because one stuck cell can fool a maximum.
EMPTY_MAX_MEAN_BYTE = float(os.environ.get("EMPTY_MAX_MEAN_BYTE", 2))

model = keras.models.load_model(MODEL_DIR / "posture.keras")
meta = js

What happens to one request:

1. The server sends the last 30 frames (100 bytes each).
2. Wrong shape → error 422. A wrong shape is never guessed at.
3. **Empty rule first:** if the average byte over the newest 5 frames is below `EMPTY_MAX_MEAN_BYTE`, answer "empty". This runs before scaling, because scaling turns noise into a fake picture (Lesson 2). It uses the average, not the hottest cell, because Lesson 2 found an empty frame with one cell at full scale. The value 2 is a **placeholder** and must be measured on our mat.
4. Flip the columns with `mat_to_model` (Lesson 2) and scale with the same `normalise_window` as in training.
5. Model → logits → softmax → answer.

The service does not care which model is in the file. Swapping models needs no code change.

We test it with FastAPI's test client, which calls the app directly without opening a port:

In [8]:
from fastapi.testclient import TestClient
import serve

client = TestClient(serve.app)
print("GET /health ->", client.get("/health").json())

def as_bytes(window):
    """A training-view window (30, 10, 10) -> the 30 x 100 bytes the mat would send."""
    return np.round(D.mat_to_model(window) * 200).reshape(30, 100).tolist()

for label in range(3):
    i = int(np.flatnonzero(y == label)[0])
    r = client.post("/predict", json={"frames": as_bytes(X[i])})
    print(f"true {D.CLASSES[label]:7s} -> {r.status_code} {r.json()}")

silent = np.full((30, 100), 1.0).tolist()
print("near-empty mat ->", client.post("/predict", json={"frames": silent}).json())
print("wrong shape    ->", client.post("/predict", json={"frames": [[0] * 64] * 30}).status_code)

t0 = time.time()
for _ in range(20):
    client.post("/predict", json={"frames": as_bytes(X[0])})
print(f"time per request: {(time.time() - t0) / 20 * 1000:.0f} ms")

D:\Projects\Perfusense\repo\ai\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


GET /health -> {'ok': True, 'model': 'posture-last-frames-v1', 'classes': ['supine', 'left', 'right'], 'tested_on_our_mat': False}
true supine  -> 200 {'posture': 'supine', 'confidence': 0.939, 'probabilities': {'supine': 0.939, 'left': 0.035, 'right': 0.026}}


true left    -> 200 {'posture': 'left', 'confidence': 0.874, 'probabilities': {'supine': 0.072, 'left': 0.874, 'right': 0.054}}
true right   -> 200 {'posture': 'right', 'confidence': 0.753, 'probabilities': {'supine': 0.118, 'left': 0.129, 'right': 0.753}}
near-empty mat -> {'posture': 'empty', 'confidence': 1.0, 'probabilities': {'empty': 1.0}}
wrong shape    -> 422


time per request: 93 ms


These three windows come from people the model trained on, so they show that the **plumbing works**, not how accurate the model is. The real accuracy is the 84.7 % from Step 3.

## How to run it

From the `ai` folder:

```bash
.venv/Scripts/python.exe -m uvicorn serve:app --port 8000
```

Then `http://localhost:8000/docs` shows a page where you can try it.

## The whole path, start to end

```
ESP32 mat (10×10, 2 frames/s)
   -> Express server: keeps 1 frame per second, the last 30 frames        [to build]
   -> POST /predict  (ai/serve.py)                                         [done]
        empty rule -> flip -> normalise_window -> model -> posture         [done]
   -> Express server: compare postures over time, hold time -> turn event  [to build, Lesson 4 logic]
   -> dashboard posture timeline                                           [to connect]
```

## What is still missing

1. **Testing on our own mat.** Every accuracy so far is on hospital mats. Record people on our mat, check the left/right direction, measure the empty threshold, then fine-tune and report that number separately.
2. **Connecting the server.** Express must send the frames and store the answer. This is a separate change to `server/`.
3. **Real patients.** All data so far comes from healthy volunteers.

## Your turn

1. Start the service and send a request from `http://localhost:8000/docs`.
2. Add a `/version` route that returns `posture.json`.
3. Write the turn logic from Lesson 4 as a small class that receives one posture per second.
4. Give the synthetic windows fake turns: join the first part of one map to the last part of another. Does the pre-trained Transformer recover on moving windows?